In [ ]:
"""
Document Processor Module
Handles markdown document ingestion and processing
"""

from pathlib import Path
import markdown
import re


class DocumentProcessor:
    """Processes markdown documents from the knowledge base"""

    def __init__(self):
        self.documents = []

    def load_markdown_file(self, file_path):
        """Load and parse a single markdown file"""
        path = Path(file_path)

        if not path.exists():
            print(f"Warning: File not found - {file_path}")
            return None

        raw_text = path.read_text(encoding="utf-8")
        html_content = markdown.markdown(raw_text)
        sections = self._extract_sections(raw_text)

        document = {
            "file_name": path.name,
            "file_path": str(path),
            "raw_text": raw_text,
            "html": html_content,
            "sections": sections
        }

        self.documents.append(document)
        return document

    def load_folder(self, folder_path):
        """Load all markdown files from a folder recursively"""
        folder = Path(folder_path)
        loaded = []

        if not folder.exists():
            print(f"Warning: Folder not found - {folder_path}")
            return loaded

        for file in sorted(folder.rglob("*.md")):
            doc = self.load_markdown_file(file)
            if doc:
                loaded.append(doc)

        return loaded

    def _extract_sections(self, text):
        """Extract sections based on markdown headers"""
        sections = {}
        current_header = "Introduction"
        current_content = []

        for line in text.split("\n"):
            if line.startswith("#"):
                if current_content:
                    sections[current_header] = "\n".join(current_content).strip()
                current_header = line.lstrip("#").strip()
                current_content = []
            else:
                current_content.append(line)

        if current_content:
            sections[current_header] = "\n".join(current_content).strip()

        return sections

    def get_all_text(self):
        """Get combined raw text from all loaded documents"""
        return "\n\n".join(doc["raw_text"] for doc in self.documents)

    def search_documents(self, keyword):
        """Search documents for a keyword"""
        results = []
        for doc in self.documents:
            if keyword.lower() in doc["raw_text"].lower():
                results.append({
                    "file_name": doc["file_name"],
                    "matches": [
                        line.strip()
                        for line in doc["raw_text"].split("\n")
                        if keyword.lower() in line.lower()
                    ]
                })
        return results

    def get_document_summary(self):
        """Get a summary of all loaded documents"""
        summary = []
        for doc in self.documents:
            summary.append({
                "file_name": doc["file_name"],
                "sections": list(doc["sections"].keys()),
                "word_count": len(doc["raw_text"].split())
            })
        return summary